# 10年定着予測 - CatBoost(L_v2) × 線形モデル(combo_all_rejected) アンサンブル

**背景**: 現在の総合最良は`28_relocation_mismatch_extended_extraction`のCatBoost + ブロックL_v2
（転居×勤務地マッチ交互作用、抽出拡張版、Public 0.529454）。一方`30_catboost_encoding_variants`・
`31_linear_model_rejected_features`で、専攻職種ミスマッチ(M)等の却下済みブロックは**CatBoostには
効かないが線形モデルには効く**ことを確認した（`31_`のロジスティック回帰でcombo_all_rejected
[F+G+H+I+K+M]がbaseline比-0.0116という最大の改善を示した）。

この2つのモデルは異なる情報源（CatBoost=L_v2の勤務地系交互作用＋月次行動パターン、
線形モデル=専攻職種ミスマッチ等のGBDT非親和的な交互作用）を捉えている可能性が高いため、
固定重みでのアンサンブルを検証する。**学習された重み（Stacking/Optimized Weighted）はOOFに
過学習しPublicで悪化するという教訓（[[ensemble_oof_overfitting]]）を踏まえ、重みはCV最適化せず、
少数の固定比率を試して選ぶ。**

## 検証方法

1. CatBoost: `28_`と同じ構成（18_ベースライン + TF-IDF A_v1 + D_expanded + ブロックL_v2）
2. 線形モデル: `31_`と同じ構成（18_ベースライン + TF-IDF A_v1 + D_expanded + combo_all_rejected
   [F+G+H+I+K+M]、One-Hot+標準化+中央値補完、L2正則化ロジスティック回帰）
3. 同一の時系列split（80/20・75/25）で両モデルを学習し、検証データに対する予測確率を
   固定比率（CatBoost 100%/95%/90%/85%/80%/70%）で加重平均し、Log Lossを比較する
4. 最もバランスの良い比率をPublicで確認する

## 実行環境
Google Colab（CPU、ハイメモリ推奨）を想定。


In [ ]:
!pip install -q catboost optuna

In [ ]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

In [ ]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

In [ ]:
import datetime
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, LogisticRegressionCV
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
SCRIPT_NAME = "32_ensemble_catboost_linear"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")

In [ ]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

## 1. 基本特徴量関数の定義（split非依存、`18_`〜`26_`と同一ロジック）

In [ ]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

In [ ]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

## 2. テキストTF-IDF（A_v1、`18_`と同一・継続採用）

In [ ]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

## 3. 四半期/加速度特徴量（D_expanded、`18_`の勝者を継続採用）

`18_`のステップAで、D_expanded（16指標）がD_original（6指標）・Dなしより2 split平均で最良と判明したため、
以降は常にD_expandedを使う（今回はDブロックの再比較は行わない）。

In [ ]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

## 4. Persona単位の基本特徴量（split非依存、`18_`と同一）

In [ ]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、v1=`27_`のPublic確認済み版 / v2=抽出拡張版）

`転居許容`フラグの抽出ロジックは`27_`と同一。`希望勤務地`の抽出のみ2種類を用意する：

- **v1**: `27_`・`25_`・`data_exploration_v3/v4/v5`と同一の正規表現（Public 0.529672で確認済み）
- **v2**: v1に加え、「◯◯を希望。」「◯◯勤務を希望。」「◯◯での勤務を希望。」パターンを追加で
  拾う拡張版。未抽出だった152件（train）を目視確認して発見した言い回し。カバー率が
  88.6%→94.1%（train）/ 95.0%（test）に向上し、ダブル悪条件の該当件数も342→354件に増加、
  効果量はp=1.4×10⁻³⁹→1.7×10⁻⁴³・オッズ比0.189→0.178とむしろ強まった。

In [ ]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    return m.group(1).strip() if m else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    # reloc_ok_rawはobject dtype(True/False/None混在)のため、~演算子は使わず
    # 明示的な等価比較でTrue/False/欠損を扱う（欠損に対する~はTypeErrorになる）
    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    # ダブル悪条件フラグ（EDA v5で確認した最も強いシグナル: 転居許容せず AND 勤務地不一致）
    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")
print("L_v1 ダブル悪条件:")
print(train_reloc_v1["転居x勤務地_ダブル悪条件_v1"].value_counts())
print("\nL_v2 ダブル悪条件:")
print(train_reloc_v2["転居x勤務地_ダブル悪条件_v2"].value_counts())

## 6. 却下済みブロックの特徴量生成（G・H・I・K・M、線形モデル用）

`20_`（G）・`21_`（H）・`24_`（I）・`25_`/`26_`（K）・`29_`（M）の検証済み実装をそのまま再現する
（Fはsplit依存のためprepare_split内部で扱う）。線形モデル側（combo_all_rejected）でのみ使用し、
CatBoost側（L_v2構成）には追加しない。

In [ ]:
def parse_all_study_themes(s):
    if pd.isna(s) or s == "受講なし":
        return [], 0.0
    parts = str(s).split("｜")
    themes, total = [], 0.0
    for part in parts:
        m = re.match(r"(.+?)：([\d.]+)時間", part)
        if m:
            themes.append(m.group(1))
            total += float(m.group(2))
    return themes, total


def create_self_study_features(monthly_df, employee_ids):
    '''自己学習実施月数・合計時間・ユニークテーマ数を集計（ブロックG、EDA v3 分析7と同一ロジック）'''
    df = monthly_df[["社員ID", "自己学習（詳細）"]].copy()
    parsed = df["自己学習（詳細）"].apply(parse_all_study_themes)
    df["_themes"] = parsed.apply(lambda x: x[0])
    df["_hours"] = parsed.apply(lambda x: x[1])

    total_hours = df.groupby("社員ID")["_hours"].sum()
    active_months = df[df["_hours"] > 0].groupby("社員ID").size()
    unique_themes = df.groupby("社員ID")["_themes"].apply(lambda s: len(set(t for tl in s for t in tl)))

    out = pd.DataFrame({
        "自己学習合計時間": total_hours,
        "自己学習実施月数": active_months,
        "自己学習ユニークテーマ数": unique_themes,
    })
    out = out.reindex(employee_ids).fillna(0.0).reset_index().rename(columns={"index": "社員ID"})
    return out


logger.info("自己学習（詳細）特徴量(ブロックG)を生成中...")
train_selfstudy = create_self_study_features(train_monthly, train_ids)
test_selfstudy = create_self_study_features(test_monthly, test_ids)
logger.info(f"ブロックG: Train {train_selfstudy.shape}, Test {test_selfstudy.shape}")


def create_engagement_deepdive_features(monthly_df, employee_ids):
    '''ブロックH（情報共有件数のゼロ月数等）'''
    other_eval_cols = ["360度評価_親和度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        info_vals = emp_data["情報共有件数"].values
        is_zero = info_vals == 0
        features["情報共有件数_ゼロ月数"] = int(is_zero.sum())
        max_run = cur_run = 0
        for v in is_zero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["情報共有件数_最長ゼロ連続月数"] = max_run

        trust_mean = emp_data["360度評価_信頼度"].mean()
        other_mean = emp_data[other_eval_cols].mean(axis=1).mean()
        features["360度評価_信頼度_相対偏差"] = (
            trust_mean - other_mean if pd.notna(trust_mean) and pd.notna(other_mean) else np.nan
        )

        features_list.append(features)
    return pd.DataFrame(features_list)


logger.info("エンゲージメント深掘り特徴量(ブロックH)を生成中...")
train_engagement = create_engagement_deepdive_features(train_monthly, train_ids)
test_engagement = create_engagement_deepdive_features(test_monthly, test_ids)
logger.info(f"ブロックH: Train {train_engagement.shape}, Test {test_engagement.shape}")


def extract_memo_section(text, section_name):
    if pd.isna(text):
        return None
    m = re.search(rf"{section_name}：(.+?)(?:\n|$)", text)
    return m.group(1).strip() if m else None


def create_personal_impression_features(persona_df):
    '''ブロックI（人物所見キーワード3種）'''
    obs_section = persona_df["入社時メモ"].apply(lambda t: extract_memo_section(t, "人物所見"))
    text = obs_section.fillna("")
    flex = text.apply(lambda t: any(k in t for k in ["柔軟", "切り替え", "適応"]))
    proactive = text.apply(lambda t: any(k in t for k in ["相談", "自ら", "主体的"]))
    plan = text.apply(lambda t: any(k in t for k in ["優先順位", "完了条件", "着実に"]))
    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "personal_柔軟性": flex.astype(int).values,
        "personal_主体性相談": proactive.astype(int).values,
        "personal_計画性": plan.astype(int).values,
    })


logger.info("人物所見キーワード特徴量(ブロックI)を生成中...")
train_personal = create_personal_impression_features(train_persona)
test_personal = create_personal_impression_features(test_persona)
logger.info(f"ブロックI: Train {train_personal.shape}, Test {test_personal.shape}")


def first_raise_month(g):
    g = g.sort_values("経過月数")
    salaries = g["月例給与_円"].values
    months = g["経過月数"].values
    base = salaries[0]
    for i in range(1, len(salaries)):
        if salaries[i] > base:
            return months[i]
    return np.nan


def create_raise_timing_features(monthly_df, employee_ids):
    '''ブロックK（早期昇給フラグ）'''
    raise_month = monthly_df.groupby("社員ID", group_keys=False).apply(first_raise_month, include_groups=False)
    raise_month = raise_month.reindex(employee_ids)
    early_flag = (raise_month <= 6).astype(int)
    return pd.DataFrame({
        "社員ID": employee_ids,
        "早期昇給フラグ": early_flag.values,
        "初回昇給月": raise_month.values,
    })


logger.info("早期昇給タイミング特徴量(ブロックK)を生成中...")
train_raise = create_raise_timing_features(train_monthly, train_ids)
test_raise = create_raise_timing_features(test_monthly, test_ids)
logger.info(f"ブロックK: Train {train_raise.shape}, Test {test_raise.shape}")

ANALYTICAL_MAJOR = {"情報", "理工学"}
ANALYTICAL_JOB = {"IT・エンジニアリング", "データ・商品企画・コンサルティング"}


def create_major_job_mismatch_features(persona_df):
    '''ブロックM（専攻職種の分析的適合ミスマッチ）'''
    is_analytical_major = persona_df["専攻分野"].isin(ANALYTICAL_MAJOR)
    is_analytical_job = persona_df["初期職種"].isin(ANALYTICAL_JOB)
    state = pd.Series("", index=persona_df.index)
    state[is_analytical_major & is_analytical_job] = "分析系専攻_分析系職種"
    state[is_analytical_major & ~is_analytical_job] = "分析系専攻_非分析系職種"
    state[~is_analytical_major & is_analytical_job] = "非分析系専攻_分析系職種"
    state[~is_analytical_major & ~is_analytical_job] = "非分析系専攻_非分析系職種"
    mismatch_flag = (~is_analytical_major & is_analytical_job).astype(int)
    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "専攻職種_適合状態": state.values,
        "専攻職種_分析ミスマッチ": mismatch_flag.values,
    })


logger.info("専攻×職種 分析的適合ミスマッチ特徴量(ブロックM)を生成中...")
train_majorjob = create_major_job_mismatch_features(train_persona)
test_majorjob = create_major_job_mismatch_features(test_persona)
logger.info(f"ブロックM: Train {train_majorjob.shape}, Test {test_majorjob.shape}")

print("✅ G・H・I・K・M 特徴量生成完了")

## 7. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`extra_blocks`パラメータで`{"L1"}`/`{"L2"}`/`{"F","G","H","I","K","M"}`を指定し、ベースライン
（D_expanded + TF-IDF A_v1、`18_`の構成、Eなし）に対してL_v1・L_v2のいずれかを単体で追加できるようにする。

In [ ]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None):
    '''指定した分割比率で特徴量を組み立てる。extra_blocks: {"L1","L2","F","G","H","I","K","M"}のサブセット'''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")

    if "G" in extra_blocks:
        tf = tf.merge(train_selfstudy, on=ID_COL, how="left")
        ttf = ttf.merge(test_selfstudy, on=ID_COL, how="left")

    if "H" in extra_blocks:
        tf = tf.merge(train_engagement, on=ID_COL, how="left")
        ttf = ttf.merge(test_engagement, on=ID_COL, how="left")

    if "I" in extra_blocks:
        tf = tf.merge(train_personal, on=ID_COL, how="left")
        ttf = ttf.merge(test_personal, on=ID_COL, how="left")

    if "K" in extra_blocks:
        tf = tf.merge(train_raise, on=ID_COL, how="left")
        ttf = ttf.merge(test_raise, on=ID_COL, how="left")

    if "M" in extra_blocks:
        tf = tf.merge(train_majorjob, on=ID_COL, how="left")
        ttf = ttf.merge(test_majorjob, on=ID_COL, how="left")

    if "F" in extra_blocks:
        # 中途入社者のみ、学習期間のIDで前職経験月数→初期等級_numの線形回帰をfitし、
        # 残差(経験等級_残差)を特徴量化する（リーク防止のためfitは学習期間のみ）
        mid_fit = tf[(tf[ID_COL].isin(train_period_ids)) & (tf["入社区分"] == "中途")]
        reg_f = LinearRegression().fit(mid_fit[["前職経験月数"]].values, mid_fit["初期等級_num"].values)
        for df_ in [tf, ttf]:
            is_mid = (df_["入社区分"] == "中途")
            df_["経験等級_残差"] = np.nan
            if is_mid.sum() > 0:
                df_.loc[is_mid, "経験等級_残差"] = (
                    df_.loc[is_mid, "初期等級_num"].values - reg_f.predict(df_.loc[is_mid, ["前職経験月数"]].values)
                )
            df_["is_中途"] = is_mid.astype(int)

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了")

## 8. チェックポイント機能（`18_`〜`31_`と同一）

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=["config", "n_features", "val_score", "submission_path"])

def save_checkpoint_row(result):
    df = pd.DataFrame([result])
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label]
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_score={row['val_score']:.6f}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了")

## 9. CatBoostモデル実行関数（`28_`と同一）

`18_`のステップBでCatBoostが大差で最良だったため、本ノートブックではCatBoostのみで検証する。

In [ ]:
def run_model_config(ag_train_data, ag_tuning_data, test_features, config_label, n_trials=25):
    feature_cols = [c for c in ag_train_data.columns if c not in ["入社日", TARGET_COL]]
    obj_cols = [c for c in feature_cols if ag_train_data[c].dtype == "object"]

    X_tr = ag_train_data[feature_cols].fillna(-999)
    y_tr = ag_train_data[TARGET_COL]
    X_va = ag_tuning_data[feature_cols].fillna(-999)
    y_va = ag_tuning_data[TARGET_COL]
    X_test = test_features[feature_cols].fillna(-999)

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000, "random_seed": SEED, "verbose": False,
            "cat_features": obj_cols, "early_stopping_rounds": 50, "task_type": "CPU",
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        return log_loss(y_va, model.predict_proba(X_va)[:, 1])

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    best_params = study.best_params

    final_model = cb.CatBoostClassifier(
        **best_params, iterations=3000, random_seed=SEED, verbose=False,
        cat_features=obj_cols, early_stopping_rounds=100, task_type="CPU",
    )
    final_model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
    val_preds = final_model.predict_proba(X_va)[:, 1]
    test_preds = final_model.predict_proba(X_test)[:, 1]

    val_score = log_loss(y_va, val_preds)
    sub_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    sub = pd.DataFrame({ID_COL: test_features.index, TARGET_COL: test_preds})
    sub.to_csv(sub_path, index=False, header=False)

    val_pred_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}_valpreds.npy"
    np.save(val_pred_path, val_preds)

    logger.info(f"[{config_label}] n_features={len(feature_cols)}, val_score={val_score:.6f}")
    return {
        "config": config_label, "n_features": len(feature_cols), "val_score": val_score,
        "submission_path": str(sub_path), "val_pred_path": str(val_pred_path),
    }

print("✅ run_model_config関数定義完了")

## 10. 線形モデル実行関数（`31_`と同一のL2ロジスティック回帰）

生カテゴリをOne-Hot、数値は学習データの中央値で補完（欠損フラグ付き）→標準化。
`Cs`は`31_`で判明した数値安定性の問題（弱正則化域での準分離によるlbfgs発散）を踏まえ、
1e-4〜10の範囲に絞って探索する。

In [ ]:
def run_model_config_linear(ag_train_data, ag_tuning_data, test_features, config_label):
    feature_cols = [c for c in ag_train_data.columns if c not in ["入社日", TARGET_COL]]
    obj_cols = [c for c in feature_cols if ag_train_data[c].dtype == "object"]
    num_cols = [c for c in feature_cols if c not in obj_cols]

    X_tr_raw = ag_train_data[feature_cols].copy()
    X_va_raw = ag_tuning_data[feature_cols].copy()
    X_test_raw = test_features[feature_cols].copy()
    y_tr = ag_train_data[TARGET_COL]
    y_va = ag_tuning_data[TARGET_COL]

    missing_flag_cols = [
        c for c in num_cols
        if X_tr_raw[c].isna().any() or X_va_raw[c].isna().any() or X_test_raw[c].isna().any()
    ]
    for c in missing_flag_cols:
        X_tr_raw[f"{c}_missing"] = X_tr_raw[c].isna().astype(float)
        X_va_raw[f"{c}_missing"] = X_va_raw[c].isna().astype(float)
        X_test_raw[f"{c}_missing"] = X_test_raw[c].isna().astype(float)

    medians = X_tr_raw[num_cols].median()
    X_tr_num = X_tr_raw[num_cols].fillna(medians).fillna(0.0)
    X_va_num = X_va_raw[num_cols].fillna(medians).fillna(0.0)
    X_test_num = X_test_raw[num_cols].fillna(medians).fillna(0.0)

    scaler = StandardScaler()
    X_tr_num_scaled = np.clip(scaler.fit_transform(X_tr_num), -10, 10)
    X_va_num_scaled = np.clip(scaler.transform(X_va_num), -10, 10)
    X_test_num_scaled = np.clip(scaler.transform(X_test_num), -10, 10)

    X_tr_cat = X_tr_raw[obj_cols].fillna("missing").astype(str)
    X_va_cat = X_va_raw[obj_cols].fillna("missing").astype(str)
    X_test_cat = X_test_raw[obj_cols].fillna("missing").astype(str)

    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    X_tr_cat_enc = ohe.fit_transform(X_tr_cat)
    X_va_cat_enc = ohe.transform(X_va_cat)
    X_test_cat_enc = ohe.transform(X_test_cat)

    if missing_flag_cols:
        mflag_cols = [f"{c}_missing" for c in missing_flag_cols]
        mflags_tr, mflags_va, mflags_test = X_tr_raw[mflag_cols].values, X_va_raw[mflag_cols].values, X_test_raw[mflag_cols].values
    else:
        mflags_tr = np.zeros((len(X_tr_raw), 0))
        mflags_va = np.zeros((len(X_va_raw), 0))
        mflags_test = np.zeros((len(X_test_raw), 0))

    X_tr_final = np.hstack([X_tr_num_scaled, X_tr_cat_enc, mflags_tr])
    X_va_final = np.hstack([X_va_num_scaled, X_va_cat_enc, mflags_va])
    X_test_final = np.hstack([X_test_num_scaled, X_test_cat_enc, mflags_test])

    model = LogisticRegressionCV(
        Cs=np.logspace(-4, 1, 15), cv=5, penalty="l2", scoring="neg_log_loss",
        max_iter=3000, random_state=SEED, n_jobs=-1,
    )
    model.fit(X_tr_final, y_tr)

    val_preds = model.predict_proba(X_va_final)[:, 1]
    test_preds = model.predict_proba(X_test_final)[:, 1]
    val_score = log_loss(y_va, val_preds)

    sub_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    sub = pd.DataFrame({ID_COL: test_features.index, TARGET_COL: test_preds})
    sub.to_csv(sub_path, index=False, header=False)
    val_pred_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}_valpreds.npy"
    np.save(val_pred_path, val_preds)

    logger.info(f"[{config_label}] encoded_features={X_tr_final.shape[1]}, best_C={float(model.C_[0]):.4g}, val_score={val_score:.6f}")
    return {
        "config": config_label, "val_score": val_score,
        "submission_path": str(sub_path), "val_pred_path": str(val_pred_path),
    }

print("✅ run_model_config_linear関数定義完了")

## 11. CatBoost(L_v2)と線形モデル(combo_all_rejected)をそれぞれ学習

同一の時系列split（80/20・75/25）で、CatBoost（extra_blocks={"L2"}）と線形モデル
（extra_blocks={"F","G","H","I","K","M"}）を別々に学習し、検証・テスト予測を保存する。
2つのprepare_split呼び出しは同じsplit_ratioに基づくため、行の並び（対象社員・順序）は一致する
はずである（下のセルで検証）。

In [ ]:
SPLIT_RATIOS = {"split_80_20": 0.8, "split_75_25": 0.75}

model_results = {}
for split_name, ratio in SPLIT_RATIOS.items():
    cb_label = f"{split_name}_catboost_Lv2"
    def _run_cb(ratio=ratio, config_label=cb_label):
        logger.info(f"=== {config_label} ===")
        ag_train_data, ag_tuning_data, test_features_full = prepare_split(ratio, extra_blocks={"L2"})
        return run_model_config(ag_train_data, ag_tuning_data, test_features_full, config_label, n_trials=25)
    cb_result = run_or_resume(cb_label, _run_cb)

    lr_label = f"{split_name}_linear_comboAllRejected"
    def _run_lr(ratio=ratio, config_label=lr_label):
        logger.info(f"=== {config_label} ===")
        ag_train_data, ag_tuning_data, test_features_full = prepare_split(ratio, extra_blocks={"F", "G", "H", "I", "K", "M"})
        return run_model_config_linear(ag_train_data, ag_tuning_data, test_features_full, config_label)
    lr_result = run_or_resume(lr_label, _run_lr)

    # val_ids/test_idsはリストのためcheckpoint CSVに安全に永続化できない（restore時に文字列化されて
    # 破損する）ので、checkpoint復元有無に関わらず毎回prepare_splitから作り直して整合性を検証する
    _, ag_tuning_cb, test_features_cb = prepare_split(ratio, extra_blocks={"L2"})
    _, ag_tuning_lr, test_features_lr = prepare_split(ratio, extra_blocks={"F", "G", "H", "I", "K", "M"})
    assert list(ag_tuning_cb.index) == list(ag_tuning_lr.index), f"{split_name}: 検証データの行順序が一致しません"
    assert list(test_features_cb.index) == list(test_features_lr.index), f"{split_name}: テストデータの行順序が一致しません"

    model_results[split_name] = {"catboost": cb_result, "linear": lr_result}

print("✅ CatBoost(L_v2)・線形モデル(combo_all_rejected)の学習完了、行順序の整合性を確認済み")

## 12. 固定重みブレンディング

学習された重み（Stacking/Optimized Weighted）はOOFに過学習しPublicで悪化するという教訓
（`data/output/submit_result_report.md`セクション6、[[ensemble_oof_overfitting]]）を踏まえ、
重みはCVで最適化せず、少数の固定比率（CatBoost 100/95/90/85/80/70%）を試して
検証Log Lossを比較する。

In [ ]:
WEIGHTS = [1.0, 0.95, 0.9, 0.85, 0.8, 0.7]

blend_results = []
for split_name in SPLIT_RATIOS:
    cb_result = model_results[split_name]["catboost"]
    lr_result = model_results[split_name]["linear"]
    cb_val = np.load(cb_result["val_pred_path"])
    lr_val = np.load(lr_result["val_pred_path"])

    _, ag_tuning_data, _ = prepare_split(SPLIT_RATIOS[split_name], extra_blocks={"L2"})
    y_va = ag_tuning_data[TARGET_COL].values

    for w in WEIGHTS:
        blended = w * cb_val + (1 - w) * lr_val
        score = log_loss(y_va, blended)
        blend_results.append({"split": split_name, "catboost_weight": w, "val_score": score})

blend_df = pd.DataFrame(blend_results)
blend_pivot = blend_df.pivot(index="catboost_weight", columns="split", values="val_score")
blend_pivot["mean"] = blend_pivot[["split_80_20", "split_75_25"]].mean(axis=1)
blend_pivot = blend_pivot.sort_values("mean")

logger.info("=" * 60)
logger.info("固定重みブレンディング結果")
logger.info("=" * 60)
logger.info("\n" + blend_pivot.to_string())
print("\n■ 固定重みブレンディング結果（catboost_weight=1.0はCatBoost単体）:")
print(blend_pivot.to_string())
print(f"\n(参考) 28_ L_v2_extended(CatBoost単体): Public 0.529454（現時点の最良）")

## 13. 最良ブレンドのテスト予測を保存

split_80_20で最も検証Log Lossが良かった重みを使い、テストデータへのブレンド予測を
提出ファイルとして保存する。

In [ ]:
best_weight_row = blend_df[blend_df["split"] == "split_80_20"].sort_values("val_score").iloc[0]
best_weight = best_weight_row["catboost_weight"]
logger.info(f"split_80_20で最良の重み: catboost_weight={best_weight}, val_score={best_weight_row['val_score']:.6f}")

cb_result_final = model_results["split_80_20"]["catboost"]
lr_result_final = model_results["split_80_20"]["linear"]

cb_test_df = pd.read_csv(cb_result_final["submission_path"], header=None, names=[ID_COL, "pred"])
lr_test_df = pd.read_csv(lr_result_final["submission_path"], header=None, names=[ID_COL, "pred"])
assert list(cb_test_df[ID_COL]) == list(lr_test_df[ID_COL]), "テストの社員ID順序が一致しません"

blended_test = best_weight * cb_test_df["pred"].values + (1 - best_weight) * lr_test_df["pred"].values
ensemble_sub_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_split_80_20_ensemble_w{best_weight}.csv"
pd.DataFrame({ID_COL: cb_test_df[ID_COL], TARGET_COL: blended_test}).to_csv(ensemble_sub_path, index=False, header=False)

logger.info(f"アンサンブル提出ファイル保存: {ensemble_sub_path}")
print(f"アンサンブル提出ファイル: {ensemble_sub_path}")
print(f"重み: CatBoost={best_weight}, Linear={1-best_weight:.2f}")

## 14. まとめ・次のアクション

1. 12節の固定重みブレンディング結果で、CatBoost単体(weight=1.0)を明確に上回る重みがあるか確認する。
   `28_`のCatBoost単体はPublic 0.529454が既知のため、検証Log Lossの改善幅がノイズ幅
   （GPU/CPU非決定性0.003〜0.006、[[catboost_gpu_nondeterminism]]）を超えているかを慎重に見る。
2. 改善が確認できた場合のみ、13節で保存したアンサンブル提出ファイルをKaggleに提出しPublicスコアを
   確認する。改善がノイズ幅程度、またはCatBoost単体が最良の場合は無理に提出せず、
   **現時点の最良は28_ L_v2_extended（Public 0.529454）のまま**とする。
3. 結果が出たら`data/output/submit_result_report.md`に追記し、メモリ（`best_submission_status.md`）
   も更新する。
4. アンサンブルがPublicで改善した場合、重みの微調整（0.85〜0.95の間をより細かく）や、
   線形モデル側の特徴量構成の見直し（`31_`で試していない構成）を次のステップとして検討する。

### バックログ（今回は着手しない）
- 学習された重み（Stacking/Optimized Weighted）での最適化は、過去の教訓を踏まえ見送る
- LightGBM/XGBoost等、CatBoost以外のGBDTをアンサンブルに加える案（過去に検証しCatBoost単体に
  劣ると確認済みのため優先度は低い）
